# MyDream LightGBM 기준 모델

확장 모델 비교 노트북과 같은 Google Drive 프로필 루트를 사용해 LightGBM 기준 모델을 학습합니다. `training_candidates_1min.csv`와 `alarm_candidates_1min.csv`가 준비되어 있어야 합니다. 이 결과는 확장 비교 노트북의 선택적 TFLite 테이블 모델 결과와는 별도입니다.

In [1]:
!pip -q install lightgbm pandas scikit-learn joblib

## Google Drive 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')

## 경로 설정

확장 모델 비교 노트북과 동일하게 `PROFILE_ROOT`는 Google Drive의 `/content/drive/MyDrive/mydream_latest/out/latest_fixed_wake_policy`를 사용합니다. 새 Colab 런타임이면 코드 저장소를 자동으로 복제합니다.

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/mydream-training-evaluation')
REPOSITORY_URL = 'https://github.com/sfpahsdev-mydream/mydream-training-evaluation.git'
PROFILE_ROOT = Path('/content/drive/MyDrive/mydream_latest/out/latest_fixed_wake_policy')
OUTPUT_DIR = PROFILE_ROOT / 'model_eval'
SCRIPT_PATH = CODE_ROOT / 'train_lightgbm_colab.py'

if not (CODE_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(CODE_ROOT)], check=True)

assert (PROFILE_ROOT / 'training_candidates_1min.csv').exists(), PROFILE_ROOT
assert (PROFILE_ROOT / 'alarm_candidates_1min.csv').exists(), PROFILE_ROOT
assert SCRIPT_PATH.exists(), SCRIPT_PATH

print('프로필 루트:', PROFILE_ROOT)
print('결과 폴더:', OUTPUT_DIR)
print('학습 스크립트:', SCRIPT_PATH)

## 학습 및 평가

In [ ]:
!python "{SCRIPT_PATH}" --input-dir "{PROFILE_ROOT}" --output-dir "{OUTPUT_DIR}" --validation-ratio 0.2 --test-ratio 0.2

## 지표 및 예측 샘플 확인

In [ ]:
import json
import pandas as pd

metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
for target in metrics['targets']:
    m = target['lightgbm']
    print(target['target'])
    print('  테스트 양성 비율:', target['positive_test_rate'])
    print('  정밀도:', m['precision'])
    print('  재현율:', m['recall'])
    print('  ROC AUC:', m.get('roc_auc'))

sample = pd.read_csv(OUTPUT_DIR / 'prediction_sample.csv')
sample.head(30)

결과 파일:

- `model_eval/label_deep_soon.joblib`
- `model_eval/metrics.json`
- `model_eval/threshold_report.csv`
- `model_eval/test_predictions_long.csv`
- `model_eval/alarm_predictions_long.csv`
- `model_eval/prediction_sample.csv`
- `model_eval/session_top_candidates.csv`
- `model_eval/session_recommendation_summary.csv`
- `model_eval/alarm_backtest_summary.csv`